In [1]:
import pandas_ta as pta
import numpy as np
import pandas as pd
import mplfinance as mpf
from tabulate import tabulate
from ggTrader.data.kraken.historical_data import KrakenHistoricalData
from ggTrader.indicators.Signals import Signals
import matplotlib.pyplot as plt
import seaborn as sns
import vectorbt as vbt


In [2]:
# Python
top20_kraken = [
    "BTC",  # 1 Bitcoin
    "ETH",  # 2 Ethereum
    "XRP",  # 3 XRP
    "BNB",  # 4 BNB
    "SOL",  # 5 Solana
    "TRX",  # 6 TRON
    "DOGE",  # 7 Dogecoin
    "ADA",  # 8 Cardano
    # "HYPE",  # 9 Hyperliquid
    "LINK",  # 10 Chainlink
    "BCH",  # 11 Bitcoin Cash
    "XLM",  # 12 Stellar
    "SUI",  # 13 Sui
    "HBAR",  # 14 Hedera
    "AVAX",  # 15 Avalanche
    "ZEC",  # 16 Zcash
    "LTC",  # 17 Litecoin
    "XMR",  # 18 Monero
    "SHIB",  # 19 Shiba Inu
    "TON",  # 20 Toncoin
    "CRO",  # 21 Crypto.com
    "DOT",  # 22 Polkadot
    "MNT",  # 23 Mantle
    "TAO",  # 24 Bittensor
    "UNI",  # 25 Uniswap
]

In [3]:
# Ticker
symbols = ["BTC", "ETH"]
# symbols = top20_kraken
interval = "4h"

# Time Range

end = pd.to_datetime("2025-06-30").tz_localize('UTC')
start = end - pd.Timedelta(days=30 * 6)

k = KrakenHistoricalData()

df_multi = k.get_ohlcv_df(symbols, interval=interval, start=start, end=end)
# k.use_remote("https://garygigabytes.com/kraken/parquet")  # no trailing slash
# df_multi = k.get_ohlcv_df_remote(symbols, interval=interval)

print(f"\nMultiIndex")
print(df_multi.head())

# select all close
print(df_multi.xs('close', axis=1, level=1).head())
close_k = df_multi.xs('close', axis=1, level=1)
high_k = df_multi.xs('high', axis=1, level=1)
low_k = df_multi.xs('low', axis=1, level=1)
open_k = df_multi.xs('open', axis=1, level=1)
# select only BTC
print(df_multi.xs('BTC', axis=1, level='symbol').head().to_string())

# list of tickers
print(df_multi.columns.levels[0].tolist())


MultiIndex
symbol                              BTC                              \
                                   open          high           low   
Datetime                                                              
2025-01-01 00:00:00+00:00  93370.796875  94250.000000  93282.000000   
2025-01-01 04:00:00+00:00  93625.101562  93654.500000  93327.796875   
2025-01-01 08:00:00+00:00  93518.703125  93535.703125  92588.101562   
2025-01-01 12:00:00+00:00  93278.101562  94200.000000  93278.000000   
2025-01-01 16:00:00+00:00  94189.796875  94581.203125  93603.796875   

symbol                                                                   \
                                  close        volume trades base quote   
Datetime                                                                  
2025-01-01 00:00:00+00:00  93625.101562  9.523742e+11   6035  BTC   USD   
2025-01-01 04:00:00+00:00  93518.703125  4.491399e+11   2874  BTC   USD   
2025-01-01 08:00:00+00:00  93278.101562  7.8

In [4]:
tickers = ["BTC-USD", "ETH-USD"]
data = vbt.YFData.download(tickers, start="2025-01-01", interval=interval)

close = data.get("Close")
high = data.get("High")
low = data.get("Low")

print(close.head().to_string())
print(f"\nVectorbt")
print(close.info())
print(f"\nKraken")
print(close_k.info())
adx = vbt.pandas_ta('adx').run(high, low, close, length=14).adx
adx_k = vbt.pandas_ta('adx').run(high_k, low_k, close_k, length=14).adx

print(adx.head())
print(adx_k.head())




symbol                          BTC-USD      ETH-USD
Datetime                                            
2025-01-01 08:00:00+00:00  93267.125000  3334.746826
2025-01-01 12:00:00+00:00  94211.062500  3345.247559
2025-01-01 16:00:00+00:00  94427.882812  3347.195068
2025-01-01 20:00:00+00:00  94443.523438  3353.471680
2025-01-02 00:00:00+00:00  94851.812500  3383.549316

Vectorbt
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1870 entries, 2025-01-01 08:00:00+00:00 to 2025-11-08 20:00:00+00:00
Freq: 4h
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   BTC-USD  1870 non-null   float64
 1   ETH-USD  1870 non-null   float64
dtypes: float64(2)
memory usage: 108.4 KB
None

Kraken
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1081 entries, 2025-01-01 00:00:00+00:00 to 2025-06-30 00:00:00+00:00
Freq: 4h
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   BTC    

In [5]:

close_k.index.name = "Datetime"
close_k.columns = close_k.columns.set_names("symbol")
print(f"\nVectorbt")
print(close.head())
print(f"\nKraken")
print(close_k.head())



Vectorbt
symbol                          BTC-USD      ETH-USD
Datetime                                            
2025-01-01 08:00:00+00:00  93267.125000  3334.746826
2025-01-01 12:00:00+00:00  94211.062500  3345.247559
2025-01-01 16:00:00+00:00  94427.882812  3347.195068
2025-01-01 20:00:00+00:00  94443.523438  3353.471680
2025-01-02 00:00:00+00:00  94851.812500  3383.549316

Kraken
symbol                              BTC          ETH
Datetime                                            
2025-01-01 00:00:00+00:00  93625.101562  3347.100098
2025-01-01 04:00:00+00:00  93518.703125  3338.989990
2025-01-01 08:00:00+00:00  93278.101562  3334.250000
2025-01-01 12:00:00+00:00  94189.796875  3345.020020
2025-01-01 16:00:00+00:00  94440.101562  3349.280029
